In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy qiskit-basis-constructor
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# مقدمة إلى البوابات الكسرية

import TutorialFeedback from '@site/src/components/TutorialFeedback';



*تقدير الاستخدام: أقل من 30 ثانية على معالج Heron r2 (ملاحظة: هذا تقدير فحسب. قد يختلف وقت التشغيل الفعلي.)*
## الخلفية النظرية
### البوابات الكسرية على وحدات المعالجة الكمومية من IBM
البوابات الكسرية هي بوابات كمومية ذات معاملات تُتيح التنفيذ المباشر للدوران بزوايا اعتباطية (ضمن حدود محددة)،
مما يُلغي الحاجة إلى تحليلها إلى بوابات أساسية متعددة.
من خلال استغلال التفاعلات الأصيلة بين البتات الكمومية الفيزيائية، يمكن للمستخدمين تطبيق وحدات unitaries معينة بكفاءة أعلى على الأجهزة.

تدعم وحدات المعالجة الكمومية IBM Quantum&reg; Heron البوابات الكسرية التالية:
- $R_{ZZ}(\theta)$ لقيم $0 < \theta < \pi / 2$
- $R_X(\theta)$ لأي قيمة حقيقية $\theta$

يمكن لهذه البوابات أن تُقلل بشكل ملحوظ من عمق الدوائر الكمومية ومدتها.
وهي مفيدة بصفة خاصة في التطبيقات التي تعتمد بشكل كبير على $R_{ZZ}$ و $R_X$،
مثل محاكاة هاميلتوني، وخوارزمية التحسين الكمومي التقريبي (QAOA)، وطرق النواة الكمومية.
في هذا البرنامج التعليمي، نركز على النواة الكمومية كمثال تطبيقي عملي.
### القيود
البوابات الكسرية ميزة تجريبية في الوقت الراهن وتأتي مع بعض القيود:
- يقتصر نطاق زاوية $R_{ZZ}$ على القيم في المجال $0 < \theta < \pi / 2$.
- لا يُدعم استخدام البوابات الكسرية مع [الدوائر الديناميكية](/guides/classical-feedforward-and-control-flow)، أو [تشابك Pauli](/guides/error-mitigation-and-suppression-techniques#pauli-twirling)، أو [إلغاء الأخطاء الاحتمالي](/guides/error-mitigation-and-suppression-techniques#probabilistic-error-cancellation-pec) (PEC)، أو [استقراء الضوضاء الصفري](/guides/error-mitigation-and-suppression-techniques#zero-noise-extrapolation-zne) (ZNE) (باستخدام [تضخيم الأخطاء الاحتمالي](/guides/error-mitigation-and-suppression-techniques#probabilistic-error-amplification-pea) (PEA)).

تتطلب البوابات الكسرية سير عمل مختلفاً مقارنةً بالنهج المعياري.
يشرح هذا البرنامج التعليمي كيفية التعامل مع البوابات الكسرية من خلال تطبيق عملي.

راجع ما يلي للمزيد من التفاصيل حول البوابات الكسرية.
- [البوابات الكسرية](/guides/fractional-gates)
- [متى *لا* تستخدم البوابات الكسرية](/guides/fractional-gates#when-not-to-use)
## نظرة عامة
يتبع سير العمل الخاص باستخدام البوابات الكسرية بشكل عام نمط [أنماط Qiskit](/guides/intro-to-patterns).
الفارق الرئيسي هو أن جميع زوايا RZZ يجب أن تستوفي الشرط $0 < \theta \leq \pi/2$.
ثمة نهجان لضمان استيفاء هذا الشرط.
يُركز هذا البرنامج التعليمي على النهج الثاني ويوصي به.

### 1. توليد قيم المعاملات التي تستوفي قيد زاوية RZZ
إذا كنت متأكداً من أن جميع زوايا RZZ تقع ضمن النطاق الصحيح، فيمكنك اتباع سير عمل أنماط Qiskit المعياري.
في هذه الحالة، تُمرر قيم المعاملات ببساطة كجزء من PUB. يسير سير العمل على النحو التالي.

In [1]:
import matplotlib.pyplot as plt
import numpy as np
from qiskit import QuantumCircuit, generate_preset_pass_manager
from qiskit.circuit import ParameterVector
from qiskit.circuit.library import UGate, n_local, unitary_overlap
from qiskit.transpiler import Target, PassManager
from qiskit.transpiler.passes import (
    Optimize1qGatesDecomposition,
    RemoveIdentityEquivalent,
)
from qiskit_aer.primitives import SamplerV2 as AerSampler
from qiskit_basis_constructor import DEFAULT_EQUIVALENCE_LIBRARY
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
from qiskit_ibm_runtime.transpiler.passes import FoldRzzAngle

إذا حاولت إرسال PUB يتضمن بوابة RZZ بزاوية خارج النطاق الصحيح، فستواجه رسالة خطأ من هذا القبيل:
```
'The instruction rzz is supported only for angles in the range [0, pi/2], but an angle (20.0) outside of this range has been requested; via parameter value(s) γ[0]=10.0, substituted in parameter expression 2.0*γ[0].'
```
لتجنب هذا الخطأ، ينبغي النظر في النهج الثاني الموضح أدناه.
### 2. تعيين قيم المعاملات للدوائر قبل عملية التحويل
توفر حزمة `qiskit-ibm-runtime` مرحلة تحويل متخصصة تُسمى [`FoldRzzAngle`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/transpiler-passes-fold-rzz-angle).
تقوم هذه المرحلة بتحويل الدوائر الكمومية بحيث تستوفي جميع زوايا RZZ قيد زاوية RZZ.
إذا زوّدت الـ backend إلى `generate_preset_pass_manager` أو `transpile`، يُطبّق Qiskit تلقائياً `FoldRzzAngle` على الدوائر الكمومية.
يستلزم ذلك تعيين قيم المعاملات للدوائر الكمومية قبل عملية التحويل.
يسير سير العمل على النحو التالي.

In [2]:
service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=133
)  # backend should be a heron device or later
backend_name = backend.name
backend_c = service.backend(backend_name)  # w/o fractional gates
backend_f = service.backend(
    backend_name, use_fractional_gates=True
)  # w/ fractional gates
print(f"Backend: {backend_name}")
print(f"No fractional gates: {backend_c.basis_gates}")
print(f"With fractional gates: {backend_f.basis_gates}")
if "rzz" not in backend_f.basis_gates:
    print(f"Backend {backend_name} does not support fractional gates")

Backend: ibm_marrakesh
No fractional gates: ['cz', 'id', 'rz', 'sx', 'x']
With fractional gates: ['cz', 'id', 'rx', 'rz', 'rzz', 'sx', 'x']


لاحظ أن سير العمل هذا يُكبّد تكلفة حسابية أعلى مقارنةً بالنهج الأول، إذ ينطوي على تعيين قيم المعاملات للدوائر الكمومية وتخزين الدوائر المرتبطة بالمعاملات محلياً.
علاوة على ذلك، ثمة مشكلة معروفة في Qiskit حيث قد تفشل عملية تحويل بوابات RZZ في سيناريوهات معينة. للاطلاع على حل بديل، يُرجى الرجوع إلى قسم [استكشاف الأخطاء وإصلاحها](#troubleshooting).
يوضح هذا البرنامج التعليمي كيفية استخدام البوابات الكسرية عبر النهج الثاني من خلال مثال مستوحى من طريقة النواة الكمومية.
لفهم أعمق لحالات الاستخدام الأمثل للنوى الكمومية، نوصي بقراءة [Liu, Arunachalam & Temme (2021).](https://www.nature.com/articles/s41567-021-01287-z)

يمكنك أيضاً العمل على البرنامج التعليمي [تدريب النواة الكمومية](/tutorials/quantum-kernel-training) ودرس [النوى الكمومية](/learning/courses/quantum-machine-learning/quantum-kernel-methods) في دورة تعلم الآلة الكمومية على IBM Quantum Learning.
### المتطلبات
قبل البدء في هذا البرنامج التعليمي، تأكد من تثبيت ما يلي:
- Qiskit SDK الإصدار v2.0 أو أحدث، مع دعم [التصور البصري](https://docs.quantum.ibm.com/api/qiskit/visualization)
- Qiskit Runtime الإصدار v0.37 أو أحدث (`pip install qiskit-ibm-runtime`)
- Qiskit Basis Constructor (`pip install qiskit_basis_constructor`)
### الإعداد

In [3]:
optimization_level = 2
shots = 2000
reps = 3
rng = np.random.default_rng(seed=123)

In [4]:
def my_zz_feature_map(num_qubits: int, reps: int = 1) -> QuantumCircuit:
    x = ParameterVector("x", num_qubits * reps)
    qc = QuantumCircuit(num_qubits)
    qc.h(range(num_qubits))
    for k in range(reps):
        K = k * num_qubits
        for i in range(num_qubits):
            qc.rz(x[i + K], i)
        pairs = [(i, i + 1) for i in range(num_qubits - 1)]
        for i, j in pairs[0::2] + pairs[1::2]:
            qc.rzz((np.pi - x[i + K]) * (np.pi - x[j + K]), i, j)
    return qc


def quantum_kernel(num_qubits: int, reps: int = 1) -> QuantumCircuit:
    qc = my_zz_feature_map(num_qubits, reps=reps)
    inner_product = unitary_overlap(qc, qc, "x", "y", insert_barrier=True)
    inner_product.measure_all()
    return inner_product


def random_parameters(inner_product: QuantumCircuit) -> np.ndarray:
    return np.tile(rng.random(inner_product.num_parameters // 2), 2)


def fidelity(result) -> float:
    ba = result.data.meas
    return ba.get_int_counts().get(0, 0) / ba.num_shots

Quantum kernel circuits and their corresponding parameter values are generated for systems with 4 to 40 qubits, and their fidelities are subsequently evaluated.

In [5]:
qubits = list(range(4, 12, 2))
circuits = [quantum_kernel(i, reps=reps) for i in qubits]
params = [random_parameters(circ) for circ in circuits]

## سير العمل مع البوابات الكسرية
### الخطوة 1: تعيين المدخلات الكلاسيكية إلى مسألة كمومية
#### دائرة النواة الكمومية
في هذا القسم، نستكشف دائرة النواة الكمومية باستخدام بوابات RZZ لتقديم سير العمل الخاص بالبوابات الكسرية.

نبدأ ببناء دائرة كمومية لحساب عناصر مصفوفة النواة بصورة فردية.
يتم ذلك من خلال دمج دوائر ZZ feature map مع تداخل وحدوي (unitary overlap).
تأخذ دالة النواة متجهات في فضاء الخريطة المميزة وتُعيد حاصل ضربها الداخلي كعنصر في مصفوفة النواة:
$$K(x, y) = \langle \Phi(x) | \Phi(y) \rangle,$$
حيث يمثل $|\Phi(x)\rangle$ الحالة الكمومية المعيّنة بالخريطة المميزة.

نبني يدوياً دائرة ZZ feature map باستخدام بوابات RZZ.
على الرغم من أن Qiskit يوفر `zz_feature_map` مدمجة، إلا أنها لا تدعم حالياً بوابات RZZ اعتباراً من Qiskit v2.0.2 ([انظر المشكلة](https://github.com/Qiskit/qiskit/issues/14469)).

بعد ذلك، نحسب دالة النواة لمدخلات متطابقة - على سبيل المثال، $K(x, x) = 1$.
على أجهزة الحوسبة الكمومية المشوشة، قد تكون هذه القيمة أقل من 1 بسبب الضوضاء.
نتيجة أقرب إلى 1 تُشير إلى ضوضاء أقل في التنفيذ.
في هذا البرنامج التعليمي، نُشير إلى هذه القيمة بـ *الإخلاص* (fidelity)، المُعرَّف بأنه
$$\text{fidelity} = K(x, x).$$

In [6]:
circuits[0].draw("mpl", fold=-1)

<Image src="../docs/images/tutorials/fractional-gates/extracted-outputs/b3d6341a-0.avif" alt="Output of the previous code cell" />

In the standard Qiskit patterns workflow, parameter values are typically passed to the Sampler or Estimator primitive as part of a PUB.
However, when using a backend that supports fractional gates, these parameter values must be explicitly assigned to the quantum circuit prior to transpilation.

In [7]:
b_qc = [
    circ.assign_parameters(param) for circ, param in zip(circuits, params)
]
b_qc[0].draw("mpl", fold=-1)

<Image src="../docs/images/tutorials/fractional-gates/extracted-outputs/6c9c1977-0.avif" alt="Output of the previous code cell" />

تُولَّد دوائر النواة الكمومية وقيم معاملاتها المقابلة لأنظمة تتراوح من 4 إلى 40 بتة كمومية، ثم يُقيَّم إخلاصها لاحقاً.

In [8]:
backend_f = service.backend(name=backend_name, use_fractional_gates=True)
# pm_f includes `FoldRzzAngle` pass
pm_f = generate_preset_pass_manager(
    optimization_level=optimization_level, backend=backend_f
)
pm_f.post_optimization = PassManager(
    [
        FoldRzzAngle(),
        Optimize1qGatesDecomposition(target=backend_f.target),
        RemoveIdentityEquivalent(target=backend_f.target),
    ]
)

In [9]:
t_qc_f = pm_f.run(b_qc)
print(t_qc_f[0].count_ops())
t_qc_f[0].draw("mpl", fold=-1)

OrderedDict({'rz': 35, 'rzz': 18, 'x': 13, 'rx': 9, 'measure': 4, 'barrier': 2})


<Image src="../docs/images/tutorials/fractional-gates/extracted-outputs/a18e5c70-1.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/fractional-gates/extracted-outputs/b3d6341a-0.avif)

في سير عمل أنماط Qiskit المعياري، تُمرَّر قيم المعاملات عادةً إلى Sampler أو Estimator كجزء من PUB.
غير أنه عند استخدام backend يدعم البوابات الكسرية، يجب تعيين قيم المعاملات هذه صراحةً إلى الدائرة الكمومية قبل عملية التحويل.

In [10]:
nnl_f = [qc.num_nonlocal_gates() for qc in t_qc_f]
depth_f = [qc.depth() for qc in t_qc_f]
duration_f = [
    qc.estimate_duration(backend_f.target, unit="u") for qc in t_qc_f
]

![Output of the previous code cell](../docs/images/tutorials/fractional-gates/extracted-outputs/6c9c1977-0.avif)

### الخطوة 2: تحسين المسألة لتنفيذها على الأجهزة الكمومية

نُحوّل بعد ذلك الدائرة باستخدام مدير المراحل وفق نمط Qiskit المعياري.
من خلال تزويد `generate_preset_pass_manager` بـ backend يدعم البوابات الكسرية، تُضاف تلقائياً مرحلة متخصصة تُسمى `FoldRzzAngle`.
تُعدّل هذه المرحلة الدائرة لتستوفي قيود زاوية RZZ.
ونتيجةً لذلك، تتحول بوابات RZZ ذات القيم السالبة في الشكل السابق إلى قيم موجبة، مع إضافة بعض بوابات X.

In [11]:
sampler_f = AerSampler.from_backend(backend_f)

In [12]:
job = sampler_f.run(t_qc_f, shots=shots)
print(job.job_id())

085ce928-767e-4200-93bf-3905e5411cfe


### Step 4: Post-process and return result in desired classical format

You can obtain the kernel function value $K(x, x)$ by measuring the probability of the all-zero bitstring `00...00` in the output.

In [13]:
result = job.result()
fidelity_f = [fidelity(result=res) for res in result]
print(fidelity_f)

[0.929, 0.882, 0.8645, 0.817]


![Output of the previous code cell](../docs/images/tutorials/fractional-gates/extracted-outputs/a18e5c70-1.avif)

لتقييم تأثير البوابات الكسرية، نقيس عدد البوابات غير المحلية (CZ و RZZ لهذا الـ backend)،
إلى جانب عمق الدوائر ومدتها، ثم نقارن هذه المقاييس بتلك الخاصة بسير العمل المعياري لاحقاً.

In [14]:
# step 1: map classical inputs to quantum problem
# `circuits` and `params` from the previous section are reused here

In [15]:
# step 2: optimize circuits
backend_c = service.backend(backend_name)  # w/o fractional gates
pm_c = generate_preset_pass_manager(
    optimization_level=optimization_level, backend=backend_c
)
t_qc_c = pm_c.run(circuits)
print(t_qc_c[0].count_ops())
t_qc_c[0].draw("mpl", fold=-1)

OrderedDict({'rz': 130, 'sx': 80, 'cz': 36, 'measure': 4, 'barrier': 2})


<Image src="../docs/images/tutorials/fractional-gates/extracted-outputs/a10f2d95-1.avif" alt="Output of the previous code cell" />

In [16]:
nnl_c = [qc.num_nonlocal_gates() for qc in t_qc_c]
depth_c = [qc.depth() for qc in t_qc_c]
duration_c = [
    qc.estimate_duration(backend_c.target, unit="u") for qc in t_qc_c
]

In [17]:
# step 3: execute
sampler_c = AerSampler.from_backend(backend_c)

In [18]:
job = sampler_c.run(pubs=zip(t_qc_c, params), shots=shots)
print(job.job_id())

f2cca29d-7263-4976-9e51-13a91b75c3ae


In [19]:
# step 4: post-processing
result = job.result()
fidelity_c = [fidelity(res) for res in result]
print(fidelity_c)

[0.8625, 0.7605, 0.702, 0.671]


## مقارنة سير العمل والدائرة بدون البوابات الكسرية
في هذا القسم، نعرض سير عمل أنماط Qiskit المعياري باستخدام backend لا يدعم البوابات الكسرية.
بمقارنة الدوائر المحوَّلة، ستلاحظ أن النسخة التي تستخدم البوابات الكسرية (من القسم السابق) أكثر إيجازاً من النسخة التي لا تستخدمها.

In [20]:
plt.plot(qubits, depth_c, "-o", label="no fractional gates")
plt.plot(qubits, depth_f, "-o", label="with fractional gates")
plt.xlabel("number of qubits")
plt.ylabel("depth")
plt.title("Comparison of depths")
plt.grid()
plt.legend()

<Image src="../docs/images/tutorials/fractional-gates/extracted-outputs/ef343a53-1.avif" alt="Output of the previous code cell" />

In [21]:
plt.plot(qubits, duration_c, "-o", label="no fractional gates")
plt.plot(qubits, duration_f, "-o", label="with fractional gates")
plt.xlabel("number of qubits")
plt.ylabel("duration (µs)")
plt.title("Comparison of durations")
plt.grid()
plt.legend()

<Image src="../docs/images/tutorials/fractional-gates/extracted-outputs/98bb2cd0-1.avif" alt="Output of the previous code cell" />

In [22]:
plt.plot(qubits, nnl_c, "-o", label="no fractional gates")
plt.plot(qubits, nnl_f, "-o", label="with fractional gates")
plt.xlabel("number of qubits")
plt.ylabel("number of non-local gates")
plt.title("Comparison of numbers of non-local gates")
plt.grid()
plt.legend()

<Image src="../docs/images/tutorials/fractional-gates/extracted-outputs/1383b242-1.avif" alt="Output of the previous code cell" />

In [23]:
plt.plot(qubits, fidelity_c, "-o", label="no fractional gates")
plt.plot(qubits, fidelity_f, "-o", label="with fractional gates")
plt.xlabel("number of qubits")
plt.ylabel("fidelity")
plt.title("Comparison of fidelities")
plt.grid()
plt.legend()

<Image src="../docs/images/tutorials/fractional-gates/extracted-outputs/8b4594f5-1.avif" alt="Output of the previous code cell" />

## Large-scale hardware example

In this section, we benchmark the quantum kernel workflow with and without fractional gates on quantum hardware with up to 40 qubits.

### Step 1-4 combined

The workflow follows the same structure as the small-scale example. We transpile all circuits with and without fractional gates, collect metrics, and then submit the circuits to real quantum hardware.

In [24]:
# -------------------------Step 1-------------------------
qubits = list(range(4, 44, 4))
circuits = [quantum_kernel(i, reps=reps) for i in qubits]
params = [random_parameters(circ) for circ in circuits]
b_qc = [
    circ.assign_parameters(param) for circ, param in zip(circuits, params)
]


def benchmark(b_qc, backend):
    # -------------------------Step 2-------------------------
    pm = generate_preset_pass_manager(optimization_level, backend=backend)
    if "rzz" in backend.target.operation_names:
        # workaround until https://github.com/Qiskit/qiskit-ibm-runtime/issues/2441 is resolved
        pm.post_optimization = PassManager(
            [
                FoldRzzAngle(),
                Optimize1qGatesDecomposition(target=backend.target),
                RemoveIdentityEquivalent(target=backend.target),
            ]
        )
    t_qc = pm.run(b_qc)
    nnl = [qc.num_nonlocal_gates() for qc in t_qc]
    depth = [qc.depth() for qc in t_qc]
    duration = [
        qc.estimate_duration(backend_f.target, unit="u") for qc in t_qc
    ]

    # -------------------------Step 3-------------------------
    sampler = SamplerV2(mode=backend)
    sampler.options.dynamical_decoupling.enable = True
    sampler.options.dynamical_decoupling.sequence_type = "XY4"
    sampler.options.dynamical_decoupling.skip_reset_qubits = True
    sampler.options.environment.job_tags = ["TUT_FG"]
    job = sampler.run(t_qc, shots=shots)
    job_id = job.job_id()
    return nnl, depth, duration, job_id


def postprocessing(job_id: str):
    # -------------------------Step 4-------------------------
    job = service.job(job_id)
    result = job.result()
    fidelities = [fidelity(result=res) for res in result]
    usage = job.usage()
    return fidelities, usage


backend_f = service.backend(backend_name, use_fractional_gates=True)
nnl_f, depth_f, duration_f, job_id_f = benchmark(
    b_qc, backend_f
)  # step 2 & 3
print("job id (w/ fractional gates):", job_id_f)
fidelity_f, usage_f = postprocessing(job_id_f)  # step 4

job id (w/ fractional gates): d8uasitbh0os73eqnpig


In [25]:
backend_c = service.backend(backend_name, use_fractional_gates=False)
nnl_c, depth_c, duration_c, job_id_c = benchmark(b_qc, backend_c)
print("job id (w/o fractional gates):", job_id_c)
fidelity_c, usage_c = postprocessing(job_id_c)

job id (w/o fractional gates): d8uav3lposuc738pruug


We then compare metrics.

In [26]:
plt.plot(qubits, depth_c, "-o", label="no fractional gates")
plt.plot(qubits, depth_f, "-o", label="with fractional gates")
plt.xlabel("number of qubits")
plt.ylabel("depth")
plt.title("Comparison of depths")
plt.grid()
plt.legend()

<Image src="../docs/images/tutorials/fractional-gates/extracted-outputs/b409e8d3-1.avif" alt="Output of the previous code cell" />

In [27]:
plt.plot(qubits, duration_c, "-o", label="no fractional gates")
plt.plot(qubits, duration_f, "-o", label="with fractional gates")
plt.xlabel("number of qubits")
plt.ylabel("duration (µs)")
plt.title("Comparison of durations")
plt.grid()
plt.legend()

<Image src="../docs/images/tutorials/fractional-gates/extracted-outputs/09f91f0f-1.avif" alt="Output of the previous code cell" />

In [28]:
plt.plot(qubits, nnl_c, "-o", label="no fractional gates")
plt.plot(qubits, nnl_f, "-o", label="with fractional gates")
plt.xlabel("number of qubits")
plt.ylabel("number of non-local gates")
plt.title("Comparison of numbers of non-local gates")
plt.grid()
plt.legend()

<Image src="../docs/images/tutorials/fractional-gates/extracted-outputs/c9308517-1.avif" alt="Output of the previous code cell" />

In [29]:
plt.plot(qubits, fidelity_c, "-o", label="no fractional gates")
plt.plot(qubits, fidelity_f, "-o", label="with fractional gates")
plt.xlabel("number of qubits")
plt.ylabel("fidelity")
plt.title("Comparison of fidelities")
plt.grid()
plt.legend()

<Image src="../docs/images/tutorials/fractional-gates/extracted-outputs/234731d4-1.avif" alt="Output of the previous code cell" />

We compare the QPU usage time with and without fractional gates. The results in the following cell show that the QPU usage times are almost identical.

In [30]:
print(f"no fractional gates: {usage_c} seconds")
print(f"fractional gates: {usage_f} seconds")

no fractional gates: 8 seconds
fractional gates: 8 seconds


## Advanced topic: Using only fractional RX gates

The need for the modified workflow when using fractional gates primarily stems from the restriction on RZZ gate angles.
However, if you use only the fractional RX gates and exclude the fractional RZZ gates, you can continue to follow the standard Qiskit patterns workflow.
This approach can still offer meaningful benefits, particularly in circuits that involve a large number of RX gates and U gates, by reducing the overall gate count and potentially improving performance.
In this section, we demonstrate how to optimize your circuits using only fractional RX gates, while omitting RZZ gates.

To support this, we provide a utility function that allows you to disable a specific basis gate in a Target object.
Here, we use it to disable RZZ gates.

In [31]:
def remove_instruction_from_target(target: Target, gate_name: str) -> Target:
    new_target = Target(
        description=target.description,
        num_qubits=target.num_qubits,
        dt=target.dt,
        granularity=target.granularity,
        min_length=target.min_length,
        pulse_alignment=target.pulse_alignment,
        acquire_alignment=target.acquire_alignment,
        qubit_properties=target.qubit_properties,
        concurrent_measurements=target.concurrent_measurements,
    )

    for name, qarg_map in target.items():
        if name == gate_name:
            continue
        instruction = target.operation_from_name(name)
        if qarg_map == {None: None}:
            qarg_map = None
        new_target.add_instruction(instruction, qarg_map, name=name)
    return new_target

We use a circuit consisting of U, CZ, and RZZ gates as an example.

In [32]:
qc = n_local(3, "u", "cz", "linear", reps=1)
qc.rzz(1.1, 0, 1)
qc.draw("mpl")

<Image src="../docs/images/tutorials/fractional-gates/extracted-outputs/6b812497-0.avif" alt="Output of the previous code cell" />

We first transpile the circuit for a backend that does not support fractional gates.

In [33]:
pm_c = generate_preset_pass_manager(
    optimization_level=optimization_level, backend=backend_c
)
t_qc = pm_c.run(qc)
print(t_qc.count_ops())
t_qc.draw("mpl")

OrderedDict({'rz': 23, 'sx': 16, 'cz': 4})


<Image src="../docs/images/tutorials/fractional-gates/extracted-outputs/9e8e0709-1.avif" alt="Output of the previous code cell" />

Then, we transpile the same circuit using fractional RX gates, while excluding RZZ gates.
This results in a slight reduction in the total gate count, thanks to the more efficient implementation of the RX gates.

In [34]:
backend_f = service.backend(backend_name, use_fractional_gates=True)
target = remove_instruction_from_target(backend_f.target, "rzz")
pm_f = generate_preset_pass_manager(
    optimization_level=optimization_level,
    target=target,
)
t_qc = pm_f.run(qc)
print(t_qc.count_ops())
t_qc.draw("mpl")

OrderedDict({'rz': 22, 'sx': 14, 'cz': 4, 'rx': 1})


<Image src="../docs/images/tutorials/fractional-gates/extracted-outputs/db45feb0-1.avif" alt="Output of the previous code cell" />

### Optimize U gates with fractional RX gates

In this section, we demonstrate how to optimize U gates using fractional RX gates, building on the same circuit introduced in the previous section.

We transpile the circuit using only fractional RX gates, excluding RZZ gates.
By introducing a custom decomposition rule, as shown in the following,
we can reduce the number of single-qubit gates required to implement a U gate.

This feature is currently under discussion in this [GitHub issue](https://github.com/Qiskit/qiskit/issues/13455).

In [35]:
# special decomposition rule for UGate
x = ParameterVector("x", 3)
zxz = QuantumCircuit(1)
zxz.rz(x[2] - np.pi / 2, 0)
zxz.rx(x[0], 0)
zxz.rz(x[1] + np.pi / 2, 0)
DEFAULT_EQUIVALENCE_LIBRARY.add_equivalence(UGate(x[0], x[1], x[2]), zxz)

Next, we apply the transpiler using `constructor-beta` translation provided by the `qiskit-basis-constructor` package.
As a result, the total number of gates is reduced compared to the previous transpilation.

In [36]:
pm_f = generate_preset_pass_manager(
    optimization_level=optimization_level,
    target=target,
    translation_method="constructor-beta",
)
t_qc = pm_f.run(qc)
print(t_qc.count_ops())
t_qc.draw("mpl")

OrderedDict({'rz': 16, 'rx': 9, 'cz': 4})


<Image src="../docs/images/tutorials/fractional-gates/extracted-outputs/b19aae7c-1.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/fractional-gates/extracted-outputs/8b4594f5-1.avif)

نقارن وقت استخدام وحدة المعالجة الكمومية مع البوابات الكسرية ودونها. تُظهر نتائج الخلية التالية أن أوقات استخدام وحدة المعالجة الكمومية متقاربة تقريباً.